In [6]:
import os

os.chdir("..")
import torch
import sys
torch.set_printoptions(threshold=sys.maxsize, linewidth=sys.maxsize)

In [ ]:
import torch
import pandas as pd
from fourm.data.multimodal_dataset_folder import MultiModalDatasetFolder
from fourm.data.modality_transforms import UnifiedDataTransform
from fourm.data.image_augmenter import EmptyAugmenter
from fourm.data.modality_info import MODALITY_INFO, MODALITY_TRANSFORMS
from tokenizers import Tokenizer

image_augmenter = EmptyAugmenter()
# all modalities
transforms = UnifiedDataTransform(
    transforms_dict=MODALITY_TRANSFORMS,
    image_augmenter=image_augmenter,
    resample_mode="bicubic",
    add_sizes=False,
)
data_df = pd.read_csv(
    "/scratch/bdej/cohanlon/data/labeled_sar_data.csv", sep="@", index_col="uid"
)
text_tokenizer = Tokenizer.from_file(
    "/u/cohanlon/sarformer/fourm/utils/tokenizer/trained/tokenizer_inc_nonUS_lower.json"
)
dataset = MultiModalDatasetFolder(
    root="/scratch/bdej/cohanlon/data/train",
    modalities=["tok_depth@224", "tok_rgb@224", "structured_data", "caption", "target_distribution"],
    data_df=data_df,
    tokenizer=text_tokenizer,
    max_text_tok_length=512,
    valid_ids=list(data_df.index),
    modality_info=MODALITY_INFO,
    modality_transforms=MODALITY_TRANSFORMS,
    transform=transforms,
    return_path=True,
)
sampler = torch.utils.data.SequentialSampler(dataset)
data_loader = torch.utils.data.DataLoader(
    dataset,
    sampler=sampler,
    batch_size=1,
    num_workers=0,
    pin_memory=True,
    drop_last=False,
)

In [ ]:
import os
os.chdir("..")

from fourm.data.modality_info import MODALITY_INFO
from run_training_sarformer import get_model
from argparse import Namespace

args = Namespace()
args.model = "sarformer_t_swiglu_qknorm"
args.cond_domains = ["caption"]
args.all_domains = args.cond_domains + ["target_distribution"]
args.num_train_timesteps = 1000
args.beta_schedule = "linear"
args.thresholding = False
args.zero_terminal_snr = True

model = get_model(args, MODALITY_INFO)
model

In [ ]:
from PIL import Image
import torchvision.transforms.functional as TF
from matplotlib import pyplot as plt

im = Image.open("/scratch/bdej/cohanlon/unlabeled/train/rgb/million-case/220683.jpg")
im1 = TF.rotate(im, 45.4, Image.Resampling.NEAREST)
im2 = TF.rotate(im, 45.4, Image.Resampling.BILINEAR)
im3 = TF.rotate(im, 45.4, Image.Resampling.BICUBIC)

plt.figure()
f, axarr = plt.subplots(1, 3)
axarr[0].imshow(im1)
axarr[1].imshow(im2)
axarr[2].imshow(im3)

plt.show()

In [ ]:
from fourm.data.modality_transforms import RGBTransform
from fourm.data.modality_info import MODALITY_INFO
import torch


def setup_modality_info(args):
    """Sets up the modality info dictionary for the given domains."""
    modality_info = {mod: MODALITY_INFO[mod] for mod in args}
    return modality_info


modality_info = setup_modality_info(["rgb"])
modality_paths = {"rgb": "rgb"}

MODALITY_TRANSFORMS_VQVAE = {}
MODALITY_TRANSFORMS_VQVAE["rgb"] = RGBTransform(
    mean_and_std="naip", color_jitter=False, no_data_value=0
)

image_augmenter_train = RandomRotationImageAugmenter(near_orthogonal=True)

transforms_train = UnifiedDataTransform(
    transforms_dict=MODALITY_TRANSFORMS_VQVAE,
    image_augmenter=image_augmenter_train,
    resample_mode="bicubic",
    add_sizes=False,
)

dataset_train = MultiModalDatasetFolder(
    root="/scratch/bdej/cohanlon/unlabeled/train",
    modalities=["rgb", "mask_valid"],
    modality_paths=modality_paths,
    modality_transforms=MODALITY_TRANSFORMS_VQVAE,
    modality_info=modality_info,
    transform=transforms_train,
    cache=False,
)

sampler_train = torch.utils.data.DistributedSampler(
    dataset_train,
    num_replicas=1,
    rank=0,
    shuffle=True,
    drop_last=True,
)

data_loader_train = torch.utils.data.DataLoader(
    dataset_train,
    sampler=sampler_train,
    batch_size=64,
    num_workers=1,
    pin_memory=False,
    drop_last=True,
)

In [ ]:
import time

start = time.time()
for i in range(4):
    next(iter(data_loader_train))
print(time.time() - start)


In [ ]:
dataset_train.class_to_idx

In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from fourm.data.modality_transforms import DepthTransform

depth_transform = DepthTransform(no_data_value=-9999.0)

# im = Image.open(
#     "/scratch/bdej/cohanlon/data/train/depth/million-case/DEM_270091.tif"
# )
im = Image.open(
    "/scratch/bdej/cohanlon/data/train/depth/million-case/DEM_901.tif"
)
t = depth_transform.image_augment(im, None, None, 0, None, None, None, "bilinear")
t = depth_transform.postprocess(t)
t = depth_transform.depth_minmax_scaling(t)

t_prime = t.flatten()
mean = t_prime.mean().item()
std = t_prime.std().item()

print(t_prime.mode()[0].item(), t_prime.min().item(), t_prime.max().item(), t_prime.mean().item(), t_prime.std().item())
plt.hist(t.flatten(), log=True, bins=100)
# t = depth_transform.truncated_depth_standardization(t)
plt.show()

In [ ]:
from scipy.stats import norm
plt.plot(np.linspace(-178, 178, 100), norm.pdf(np.linspace(-178, 178, 100), loc=mean, scale=std))
plt.show()

In [ ]:
from PIL import Image
import numpy as np
import torchvision.transforms.functional as TF
im = Image.open("/scratch/bdej/cohanlon/data/train/rgb/million-case/90181.jpg")
TF.to_tensor(im).shape

In [ ]:
import numpy as np
import torch

test = np.array([[-np.inf, 2, 3], [4, 5, np.inf], [-9999.0, np.nan, 9]])
test = np.expand_dims(test, 0)
test

In [ ]:
test = torch.tensor(test)
test

In [ ]:
no_data_mods = ["depth"]
sample_dict = {"depth": test}
modality_info = {"depth": {"no_data_value": -9999.0, "num_channels": 1}}

channel_dim = 0
H, W = sample_dict[no_data_mods[0]].shape[1:3]
mask = torch.ones(1, H, W, dtype=torch.bool)
for mod in no_data_mods:
    sample = sample_dict[mod]
    assert sample.shape[channel_dim] == modality_info[mod]["num_channels"]
    no_data_value = modality_info[mod]["no_data_value"]

    no_data_mask = (sample != no_data_value).all(dim=channel_dim, keepdim=True)
    nan_mask = np.isfinite(sample).any(dim=channel_dim, keepdim=True)
    mask = mask & no_data_mask & nan_mask

mask

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from scipy.stats import norm

arr = np.load("../clean_images.npy")
arr = arr[:, 0].flatten()
arr = arr[arr != 0]
mean = arr.mean()
std =  arr.std()


# def reject_outliers(data, m=2.0):
#     d = np.abs(data - np.median(data))
#     mdev = np.median(d)
#     s = d / mdev if mdev else np.zeros(len(d))
#     return data[s < m]

# arr = reject_outliers(arr, 3)

plt.hist(arr, bins=1000, log=True, density=False)
plt.show()
arr.mean(), arr.std(), arr.min(), arr.max()

In [ ]:
arr_trim = arr[(arr > mean - 3 * std) & (arr < mean + 3 * std)]

plt.hist(arr_trim, bins=1000, log=True, density=False)
plt.show()
arr_trim.mean(), arr_trim.std(), arr_trim.min(), arr_trim.max()

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from PIL import Image
from scipy.stats import mode, iqr
import torch
arr = np.array(Image.open("/scratch/bdej/cohanlon/data/train/depth/million-case/DEM_79.tif"))

from fourm.data.modality_transforms import DepthTransform

arr = np.expand_dims(arr, axis=0)

arr = arr.transpose(1, 2, 0)
arr = torch.tensor(arr)

mask = DepthTransform.depth_artifact_mask(arr, outlier_threshold=16)

# arr = DepthTransform.depth_robust_scaling(arr)

arr = DepthTransform.depth_minmax_scaling(arr)

arr[~mask] = 0

# # Robust scaling
# arr[arr == -9999.0] = 0
# arr = (arr - np.median(arr[arr != 0])) / iqr(arr[arr != 0])

# # Robust trimming
# dists_from_median = np.abs(arr - np.median(arr[arr != 0]))
# dists_in_iqrs = dists_from_median / iqr(arr[arr != 0])
# outlier_threshold = 7
# # make removed pix (the outliers) black
# arr[dists_in_iqrs > outlier_threshold] = 0

d_arr = arr[arr != 0].numpy().flatten()
print(mode(d_arr).mode, d_arr.mean(), np.median(d_arr), d_arr.std(), d_arr.min(), d_arr.max())
plt.hist(d_arr, bins=1000, log=True, density=False)
plt.show()

# make kept pix white
# arr[dists_in_iqrs <= outlier_threshold] = 1

# arr[mask] = 1

print("Remaining:", round(len(arr[mask]) / 224 ** 2 * 100, 2), "%") 

arr = (arr.numpy() * 255).astype(np.uint8).repeat(3, axis=2)
im = Image.fromarray(arr)
im

In [ ]:
import numpy as np
import torch
import os
from pathlib import Path

root = Path("/scratch/bdej/cohanlon/data/train/rgb_toks/million-case")

arr = np.load(root / "200683.npy")
arr

In [ ]:
import torch
from einops import repeat
tokens = torch.arange(0, 16).reshape(4, 4).unsqueeze(0)
tokens

In [ ]:
mask = torch.tensor([[0, 1, 1, 0]])
mask_arange = (torch.arange(0, 4) * 1e-6).unsqueeze(0)
ids_shuffle = torch.argsort(mask + mask_arange, dim=1)[:, :4]
print(ids_shuffle)
# torch.gather(tokens, dim=1, index=repeat(ids_shuffle, "b n -> b n d", d=tokens.shape[-1]))
# ids = repeat(ids_shuffle, "b n -> b n d", d=tokens.shape[-1])
tokens[:, [3, 0], :]

In [ ]:
import os

os.chdir("..")
import torch
import torch.nn as nn
from fourm.models.unet import PatchedConvNeXtUNet

# TODO: we need to control for params and depth
# should be near patched unet base
model = PatchedConvNeXtUNet(
    in_channels=1,
    out_channels=1,
    num_heads=8,
    cond_dim=768,
    act_layer=nn.SiLU,
    patch_size=4,
    model_channels=128,
    num_conv_blocks=3,
    channel_mult=(1, 2, 4, 8, 16),
)
print(f"# params: {sum(p.numel() for p in model.parameters()):_}")
model

In [ ]:
x = torch.randn(5, 1, 224, 224)
timesteps = torch.randint(10, size=(5,))
cond = torch.randn(5, 1, 768)
out = model(x, timesteps, cond)
out.shape

In [ ]:
from fourm.models.sarformer import sarformer_t_swiglu_qknorm_nobias

model = sarformer_t_swiglu_qknorm_nobias(encoder_embeddings=None)
print(f"{model.get_num_encoder_params():_}, {model.get_num_backbone_params():_}")

In [ ]:
import torch
from fourm.models.encoder_embeddings import ImageTokenEncoderEmbedding
from fourm.models.sarformer import sarformer_t_swiglu_qknorm
from fourm.data.modality_info import MODALITY_INFO

encoder_embeddings = {
    "tok_rgb@224": ImageTokenEncoderEmbedding(
        vocab_size=16384, patch_size=16, dim_tokens=768, image_size=224
    )
}

mod_dict = {"tok_rgb@224": {"tensor": torch.randint(16385, size=(5, 14, 14))}}
noisy_image = torch.randn(5, 1, 224, 224)
timesteps = torch.randint(10, size=(5,))

model = sarformer_t_swiglu_qknorm(
    encoder_embeddings=encoder_embeddings,
    modality_info=MODALITY_INFO,
)
print(model.get_num_encoder_params())
model

In [4]:
from tokenizers import Tokenizer
from copy import deepcopy
import numpy as np
tokenizer = Tokenizer.from_file("../fourm/utils/tokenizer/trained/tokenizer_inc_nonUS_lower.json")

tokenizer.enable_padding(length=512)
tokenizer.enable_truncation(max_length=512)

In [ ]:
import random
import torch
import numpy as np
from PIL import Image
import torch.nn.functional as F
from einops import rearrange, repeat


def spatial_softmax(x):
    H, W = x.shape[-2:]
    x = rearrange(x, "... c h w -> ... c (h w)")
    x = F.softmax(x, dim=-1)
    return rearrange(x, "... c (h w) -> ... c h w", h=H, w=W)


def create_overlaid_img(
    target_dist: torch.Tensor, pred_dist: torch.Tensor, original_img_path: str
):
    """Creates an image with the target and predictions overlaid on the original image.
    The target distribution's single non-zero value is replaced by a black square to make it more visible.
    All distributions should sum to 1.

    Args:
        target_dist: The target distribution. Shape (1, H, W)
        pred_dist: The predicted distribution. Shape (1, H, W)
        original_img_path: Path to the original image.
    """
    pred_alpha_mask = (
        255 * rearrange(pred_dist, "1 h w -> h w 1").float().cpu().numpy()
    ).astype(np.uint8)

    H, W = target_dist.shape[-2:]

    # Byte array that corresponds to a yellow image
    yellow = np.concatenate(
        [255 * np.ones((H, W, 2)), np.zeros((H, W, 1))], axis=-1
    ).astype(np.uint8)

    pred_yellow_img = Image.fromarray(
        np.concatenate([yellow, pred_alpha_mask], axis=-1), mode="RGBA"
    )

    # White image with completely transparent alpha channel
    target_img_arr = np.concatenate(
        [255 * np.ones((H, W, 3)), np.zeros((H, W, 1))], axis=-1
    ).astype(np.uint8)

    # (i, j) of the single non-zero value in the target distribution
    i = np.argmax(target_dist, axis=-2).max().item()
    j = np.argmax(target_dist, axis=-1).max().item()

    margin = 1  # Pixel margin for black square
    lower_i = i - (margin if i - margin >= 0 else 0)
    upper_i = i + (1 + margin if i + margin < H else 1)
    lower_j = j - (margin if j - margin >= 0 else 0)
    upper_j = j + (1 + margin if j + margin < W else 1)

    # Create black square at the position of the 1 in the target distribution
    target_img_arr[lower_i:upper_i, lower_j:upper_j, :-1] = 0
    # Set the black square to full opacity
    target_img_arr[lower_i:upper_i, lower_j:upper_j, -1] = 255

    target_img = Image.fromarray(target_img_arr, mode="RGBA")

    im = Image.open(original_img_path)
    im.paste(pred_yellow_img, (0, 0), pred_yellow_img)
    im.paste(target_img, (0, 0), target_img)

    return im


t = torch.zeros(1, 224, 224)
i, j = random.randint(0, 223), random.randint(0, 223)
t[:, i, j] = 1

p = torch.randn(1, 224, 224)
# p = spatial_softmax(p)

create_overlaid_img(
    t, p, "/scratch/bdej/cohanlon/data/train/rgb/labeled_sar/NYLabeled1451.tif"
)

In [ ]:
import torch
from einops import repeat

t = torch.tensor([[0, 0], [0, 1]])
a = t.argmax(dim=-2).max().item()
b = t.argmax(dim=-1).max().item()
print(a, b)

# H, W = 2, 2
# i = repeat(torch.arange(H), "h -> 1 h w", w=W)
# j = repeat(torch.arange(W), "w -> 1 h w", h=H)

# dists = torch.sqrt((i - a) ** 2 + (j - b) ** 2)
# dists.shape


In [ ]:
import os

os.chdir("..")
from run_training_sarformer import distance_weighted_loss
import torch

logits = torch.randn(1, 1, 224, 224)
target = torch.zeros_like(logits)
target[:, :, 112, 112] = 1

distance_weighted_loss(logits, target, "bce", "euclidean")

In [1]:
from transformers import T5EncoderModel, T5Tokenizer

